# Training the CNN model

Cargamos los splits de datos (train/val/test) junto con los metadatos del dataset, incluyendo los pesos de clase para manejar el desbalance.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.utils.config import CLASSES
from src.data.loader import load_splits

df_train, df_val, df_test, meta = load_splits(ROOT, classes=CLASSES)
print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
print("Class weights:", meta["class_weight_int"])

Creamos el pipeline tf.data, construimos la CNN, entrenamos con early stopping y guardamos el mejor modelo. Luego generamos predicciones en validación y test.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from src.utils.config import SEED, CLASSES, CNN_IMG_SIZE, MODELS_DIR, RESULTS_DIR
from src.utils.evaluation import save_predictions
from src.models.cnn import build_cnn_model

tf.keras.utils.set_random_seed(SEED)
IMG_SIZE = CNN_IMG_SIZE
BATCH = 32

def make_ds(frame, shuffle):
    def load(path, y):
        img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        return tf.image.resize(img, IMG_SIZE), y
    ds = tf.data.Dataset.from_tensor_slices((frame.path.values, frame.label.values))
    if shuffle:
        ds = ds.shuffle(len(frame), seed=SEED)
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

ds_train = make_ds(df_train, shuffle=True)
ds_val = make_ds(df_val, shuffle=False)
ds_test = make_ds(df_test, shuffle=False)

model = build_cnn_model(input_shape=(*IMG_SIZE, 3), num_classes=len(CLASSES), augment=False)
model.summary()

from sklearn.utils.class_weight import compute_class_weight

class_weight = meta["class_weight_int"]        
MODELS_DIR.mkdir(parents=True, exist_ok=True)   

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint(MODELS_DIR / "cnn_best.keras", monitor="val_loss", save_best_only=True),
]
history = model.fit(ds_train, validation_data=ds_val, epochs=25,
                    class_weight=class_weight, callbacks=callbacks)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(history.history).to_csv(RESULTS_DIR / "cnn_history.csv", index=False)

for split_name, ds, df in [("val", ds_val, df_val), ("test", ds_test, df_test)]:
    proba = model.predict(ds, verbose=0)
    y_pred = [CLASSES[i] for i in proba.argmax(axis=1)]
    y_true = [CLASSES[i] for i in df.label]
    save_predictions(df.path.values, y_true, y_pred, proba, model_name="cnn", split=split_name)